In [ ]:
# ViT vs CNN

# Loading ViT model
from transformers import ViTForImageClassification, ViTImageProcessor

model_name = "google/vit-base-patch16-224"

processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=10
)

In [5]:
# Loading Dataset
from datasets import load_dataset

dataset = load_dataset("cifar10")
print(dataset)

# Lets start with only 1000 samples :D
train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"]

print(train_data)
print(test_data)

DatasetDict({
    train: Dataset({
        features: ['img', 'label'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['img', 'label'],
        num_rows: 10000
    })
})
Dataset({
    features: ['img', 'label'],
    num_rows: 1000
})
Dataset({
    features: ['img', 'label'],
    num_rows: 10000
})


In [6]:
# We need to preprocess the images for the ViT model
def preprocess_images(examples):
    inputs = processor(
        examples['img'],
        return_tensors="pt"
    )

    inputs['labels'] = examples['label']

    return inputs

train_data = train_data.with_transform(preprocess_images)
test_data = test_data.with_transform(preprocess_images)

In [ ]:
# Fine-tuning ViT model
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./vit-small",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True, # Enabling AMP, for faster GPU training
    dataloader_num_workers=4 # Using multiple workers for data loading, also for faster training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.701554
2,1.089400,0.444270
3,1.089400,0.305222
4,0.207600,0.305479
5,0.080400,0.280448
6,0.080400,0.273315
7,0.050900,0.264394
8,0.042200,0.243092
9,0.042200,0.239116
10,0.034200,0.238512


TrainOutput(global_step=320, training_loss=0.23715997841209174, metrics={'train_runtime': 2754.2549, 'train_samples_per_second': 3.631, 'train_steps_per_second': 0.116, 'total_flos': 7.7497545904128e+17, 'train_loss': 0.23715997841209174, 'epoch': 10.0})

In [ ]:
# Fine-tuning ResNet model

# Loading CNN model
from torchvision import models
import torch

restnet = models.resnet18(weights=True)
restnet.fc = torch.nn.Linear(restnet.fc.in_features, 10)

for param in restnet.parameters():
    param.requires_grad = True

for param in restnet.fc.parameters():
    param.requires_grad = True

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(restnet.parameters(), lr=2e-4)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
restnet.to(device)

for epoch in range(10): # Number of epochs
    restnet.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = restnet(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")